# Data Cleansing: Assignment 2 
- Big Data - 2025
- Juan Sebastian Vargas Torres

## Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
import matplotlib.cbook as cbook
import matplotlib.pyplot as plt

## Utils

In [0]:
def summarize_nulls(input_df: DataFrame) -> DataFrame:
    """
    Calculates the count and percentage of null values for each column in a Spark DataFrame.

    The function returns a new DataFrame with three columns:
    - column_name: The name of the column from the input DataFrame.
    - null_count: The total number of null values in that column.
    - null_percentage: The percentage of nulls relative to the total number of rows.

    Args:
        input_df: The Spark DataFrame to be analyzed.

    Returns:
        A new Spark DataFrame summarizing the null values, sorted by null_percentage descending.
    """
    total_rows = input_df.count()

    if total_rows == 0:
        schema = StructType([
            StructField("column_name", StringType(), False),
            StructField("null_count", LongType(), False),
            StructField("null_percentage", F.DoubleType(), False)
        ])
        return input_df.sparkSession.createDataFrame([], schema)

    null_counts_agg = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in input_df.columns]

    null_counts_dict = input_df.agg(*null_counts_agg).collect()[0].asDict()

    summary_data = [(k, v) for k, v in null_counts_dict.items()]

    summary_schema = StructType([
        StructField("column_name", StringType(), False),
        StructField("null_count", LongType(), False)
    ])

    summary_df = input_df.sparkSession.createDataFrame(summary_data, schema=summary_schema)

    summary_df = summary_df.withColumn(
        "null_percentage",
        F.floor(((F.col("null_count") / total_rows) * 100) * 100) / 100 # Truncate to 2 decimal places
    ).orderBy(F.desc("null_percentage"))

    return summary_df

In [0]:
def check_removal_percentage(df: DataFrame, original_size: int) -> None:
    """
    Calculates the percentage of rows removed from a Spark DataFrame,
    prints a summary, and asserts that the removal is less than 10%.

    Args:
        df (DataFrame): The DataFrame after cleaning/filtering.
        original_size (int): The original number of rows before cleaning.
    """
    current_size = df.count()
    rows_removed = original_size - current_size
    
    if original_size == 0:
        percentage_removed = 0.0
    else:
        percentage_removed = (rows_removed / original_size) * 100

    print(f"--- Data Removal Report ---")
    print(f"Original size: {original_size:,}")
    print(f"Current size : {current_size:,}")
    print(f"Rows removed : {rows_removed:,}")
    print(f"Percentage removed: {percentage_removed:.2f}%")

    bar = "█" * int(percentage_removed / 2) + "-" * (50 - int(percentage_removed / 2))
    print(f"[0% |{bar}| 100%]")
    print("-" * 28)

    assert percentage_removed < 10, \
        f"Error: More than 10% of rows removed from the DataFrame!"

In [0]:
def get_boxplot_stats(df: DataFrame, column_name: str) -> dict:
    """
    Calculates and returns summary statistics for a box plot for a specified
    column in a Spark DataFrame.

    Args:
        df (DataFrame): The input Spark DataFrame.
        column_name (str): The name of the numerical column to analyze.

    Returns:
        dict: A dictionary containing the summary statistics:
              'q1', 'median', 'q3', 'iqr', 'lower_bound', 'upper_bound'.
    """
    try:
        quantiles = df.approxQuantile(column_name, [0.25, 0.5, 0.75], 0.0)
    except Exception as e:
        print(f"Error calculating quantiles for column '{column_name}': {e}")
        print("Please ensure the column is of a numerical type.")
        return {}

    q1, median, q3 = quantiles[0], quantiles[1], quantiles[2]

    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    returned_stats = {
        'column': column_name,
        'q1': q1,
        'median': median,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }

    return returned_stats

## Reading Data

In [0]:
catalog = "workspace"
source_schema  = "bde"
source_volume  = "assignment2"
target_schema = "bde_silver"

location_table_name = "location"
taxis_table_name = 'taxis_trips'

GREEN_DST = f"/Volumes/{catalog}/{source_schema}/{source_volume}/green"
YELLOW_DST = f"/Volumes/{catalog}/{source_schema}/{source_volume}/yellow"

LOCATION_TABLE_PATH = f"{catalog}.{source_schema}.{location_table_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{target_schema}")
TAXIS_TABLE_PATH = f"{catalog}.{target_schema}.{taxis_table_name}"

In [0]:
green_df  = spark.read.parquet(GREEN_DST)
yellow_df = spark.read.parquet(YELLOW_DST)

## Data Cleansing

In [0]:
original_green_size = green_df.count()
original_yellow_size = yellow_df.count()

### Filter when dropoff datetime >= the pickup datetime

In [0]:
green_df = green_df.where(
    F.col('lpep_dropoff_datetime') >= F.col('lpep_pickup_datetime')
)

yellow_df = yellow_df.where(
    F.col('tpep_dropoff_datetime') >= F.col('tpep_pickup_datetime')
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Filter when pickup datetime, dropoff datetime <= 2024

In [0]:
(green_df
    .withColumn('year', F.year(F.col('lpep_pickup_datetime')))
    .groupBy('year')
    .count()
    .orderBy('year')
    .show()
)

(yellow_df
    .withColumn('year', F.year(F.col('tpep_pickup_datetime')))
    .groupBy('year')
    .count()
    .orderBy('year')
    .show()
)

+----+--------+
|year|   count|
+----+--------+
|2008|     114|
|2009|     315|
|2010|     348|
|2012|       3|
|2014|15836807|
|2015|19233506|
|2016|16385440|
|2017|11736855|
|2018| 8899286|
|2019| 6300512|
|2020| 1734072|
|2021| 1068727|
|2022|  840393|
|2023|  787055|
|2024|  660196|
|2025|      11|
|2030|       2|
|2035|       1|
|2041|       1|
|2062|       1|
+----+--------+
only showing top 20 rows
+----+---------+
|year|    count|
+----+---------+
|2001|       27|
|2002|      498|
|2003|       50|
|2004|        1|
|2008|      770|
|2009|     1298|
|2010|        1|
|2011|        4|
|2012|        1|
|2014|165438673|
|2015|146035315|
|2016|131129625|
|2017|103203500|
|2018|102869480|
|2019| 84596238|
|2020| 24629255|
|2021| 30866197|
|2022| 39642009|
|2023| 38307663|
|2024| 41168095|
+----+---------+
only showing top 20 rows


In [0]:
(green_df
    .withColumn('year', F.year(F.col('lpep_dropoff_datetime')))
    .groupBy('year')
    .count()
    .orderBy('year')
    .show()
)

(yellow_df
    .withColumn('year', F.year(F.col('tpep_dropoff_datetime')))
    .groupBy('year')
    .count()
    .orderBy('year')
    .show()
)


+----+--------+
|year|   count|
+----+--------+
|2008|      89|
|2009|     340|
|2010|     348|
|2012|       3|
|2014|15836146|
|2015|19233038|
|2016|16385841|
|2017|11737164|
|2018| 8899435|
|2019| 6300616|
|2020| 1734224|
|2021| 1068719|
|2022|  840396|
|2023|  787063|
|2024|  660188|
|2025|      30|
|2030|       2|
|2035|       1|
|2041|       1|
|2062|       1|
+----+--------+
only showing top 20 rows
+----+---------+
|year|    count|
+----+---------+
|2001|       27|
|2002|      334|
|2003|       75|
|2004|        1|
|2008|      504|
|2009|     1564|
|2010|        1|
|2011|        4|
|2012|        1|
|2014|165436384|
|2015|146035201|
|2016|131130079|
|2017|103203631|
|2018|102869578|
|2019| 84596700|
|2020| 24630318|
|2021| 30865886|
|2022| 39642050|
|2023| 38307732|
|2024| 41168013|
+----+---------+
only showing top 20 rows


In [0]:
green_df = green_df.filter(
    (F.year(F.col('lpep_pickup_datetime')) <= 2024) & 
    (F.year(F.col('lpep_dropoff_datetime')) <= 2024)
)

yellow_df = yellow_df.filter(
    (F.year(F.col('tpep_pickup_datetime')) <= 2024) & 
    (F.year(F.col('tpep_dropoff_datetime')) <= 2024)
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Filter when velocity > 0 

#### Creating speed column

In [0]:
green_df = (
    green_df.withColumn(
        'duration_in_seconds',
        (F.col('lpep_dropoff_datetime').cast('long') -
         F.col('lpep_pickup_datetime').cast('long'))
    )
    .withColumn(
        'duration_in_hours',
        F.round(F.col('duration_in_seconds') / 3600, 2)
    )
    .withColumn(
        'duration_in_minutes',
        F.round(F.col('duration_in_seconds') / 60, 2)
    )
    .withColumn(
        'speed',
        F.when(
            F.col('duration_in_hours') != 0,
            F.round(F.col('trip_distance') / F.col('duration_in_hours'), 2)
        ).otherwise(0)
    )
    .drop('duration_in_seconds')
)

yellow_df = (
    yellow_df.withColumn(
        'duration_in_seconds',
        (F.col('tpep_dropoff_datetime').cast('long') -
         F.col('tpep_pickup_datetime').cast('long'))
    )
    .withColumn(
        'duration_in_hours',
        F.round(F.col('duration_in_seconds') / 3600, 2)
    )
    .withColumn(
        'duration_in_minutes',
        F.round(F.col('duration_in_seconds') / 60, 2)
    )
    .withColumn(
        'speed',
        F.when(
            F.col('duration_in_hours') != 0,
            F.round(F.col('trip_distance') / F.col('duration_in_hours'), 2)
        ).otherwise(0)
    )
    .drop('duration_in_seconds')
)

In [0]:
green_df = green_df.filter(
    (F.col('speed') > 0)
)

yellow_df = yellow_df.filter(
    (F.col('speed') > 0)
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Remove over speed limit records

In [0]:
nyc_speed_limit = 25
nyc_boroughs_speed_limit = 30

green_df = green_df.filter(
    F.col('speed') <= nyc_boroughs_speed_limit
)

yellow_df = yellow_df.filter(
    F.col('speed') <= nyc_speed_limit
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Remove too short, long trips (duration)

In [0]:
min_duration_minutes = 1
max_duration_minutes = 60 * 3

green_df = green_df.filter(
    (F.col('duration_in_minutes') >= min_duration_minutes) &
    (F.col('duration_in_minutes') <= max_duration_minutes)
)

yellow_df = yellow_df.filter(
    (F.col('duration_in_minutes') >= min_duration_minutes) &
    (F.col('duration_in_minutes') <= max_duration_minutes)
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Remove too short, long trips (distance)

In [0]:
min_distance_miles = 0.1 # 160 meters or 0.16 km
max_distance_miles = 40 # Distance from State Island to Bronx

green_df = green_df.filter(
    (F.col('trip_distance') >= min_distance_miles) & 
    (F.col('trip_distance') <= max_distance_miles)
)

yellow_df = yellow_df.filter(
    (F.col('trip_distance') >= min_distance_miles) & 
    (F.col('trip_distance') <= max_distance_miles)
)

# check_removal_percentage(green_df, original_green_size)
# check_removal_percentage(yellow_df, original_yellow_size)

### Drop Columns with multiple nulls

In [0]:
summarize_nulls(green_df).display()

column_name,null_count,null_percentage
ehail_fee,79644119,99.99
congestion_surcharge,71188418,89.37
improvement_surcharge,13527722,16.98
trip_type,3431016,4.3
store_and_fwd_flag,1739472,2.18
RatecodeID,1739472,2.18
passenger_count,1739472,2.18
payment_type,1739472,2.18
VendorID,0,0.0
lpep_pickup_datetime,0,0.0


In [0]:
summarize_nulls(yellow_df).display()

column_name,null_count,null_percentage
airport_fee,792496569,92.77
congestion_surcharge,625121973,73.17
store_and_fwd_flag,54906075,6.42
improvement_surcharge,50327427,5.89
passenger_count,8066673,0.94
RatecodeID,8066673,0.94
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
trip_distance,0,0.0


In [0]:
columns_to_drop_green_df = ['ehail_fee','congestion_surcharge']
columns_to_drop_yellow_df = ['airport_fee','congestion_surcharge']

green_df = green_df.drop(*columns_to_drop_green_df)
yellow_df = yellow_df.drop(*columns_to_drop_yellow_df)

### Dropping Uninformative Columns

It can be seen that 90% or records have null or 0.3, so this type of column don't provide any information for a future ML model

In [0]:
green_df.select('improvement_surcharge').groupBy('improvement_surcharge').count().display()

improvement_surcharge,count
null,13527722
0.0,3379210
0.3,61343499
-0.3,66651
0.33,16
0.77,2
0.62,3
0.39,11
1.26,1
0.42,9


In [0]:
yellow_df.select('improvement_surcharge').groupBy('improvement_surcharge').count().display()

improvement_surcharge,count
null,50327427
0.0,105718317
0.3,623180848
-0.3,534732
0.7,1
0.6,11
4000.3,1
0.45,1
0.03,9
1.0,73733977


In [0]:
columns_to_drop_green_df = ['improvement_surcharge']
columns_to_drop_yellow_df = ['improvement_surcharge']

green_df = green_df.drop(*columns_to_drop_green_df)
yellow_df = yellow_df.drop(*columns_to_drop_yellow_df)

### Delete 0 passangers trips

In [0]:
green_df = green_df.filter(F.col('passenger_count')>0)
yellow_df = yellow_df.filter(F.col('passenger_count')>0)

### Drop null values

#### Green Taxis

In [0]:
summarize_nulls(green_df).display()

column_name,null_count,null_percentage
trip_type,1691029,2.17
VendorID,0,0.0
lpep_pickup_datetime,0,0.0
lpep_dropoff_datetime,0,0.0
store_and_fwd_flag,0,0.0
RatecodeID,0,0.0
PULocationID,0,0.0
DOLocationID,0,0.0
passenger_count,0,0.0
trip_distance,0,0.0


In [0]:
green_df = green_df.filter(F.col('store_and_fwd_flag').isNotNull())
green_df = green_df.filter(F.col('RatecodeID').isNotNull())
green_df = green_df.filter(F.col('passenger_count').isNotNull())
green_df = green_df.filter(F.col('payment_type').isNotNull())

In [0]:
green_df.groupBy('trip_type').count().display()

trip_type,count
null,1691029
1.0,74913475
2.0,1255713


In [0]:
green_trip_type_mode = green_df.select(F.mode('trip_type')).first()[0]
green_trip_type_mode

1.0

In [0]:
green_df = green_df.na.fill(value=green_trip_type_mode, subset=["trip_type"])

#### Yellow Taxis

In [0]:
summarize_nulls(yellow_df).display()

column_name,null_count,null_percentage
store_and_fwd_flag,46839332,5.56
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
passenger_count,0,0.0
trip_distance,0,0.0
RatecodeID,0,0.0
PULocationID,0,0.0
DOLocationID,0,0.0
payment_type,0,0.0


In [0]:
yellow_df = yellow_df.filter(F.col('passenger_count').isNotNull())
yellow_df = yellow_df.filter(F.col('RatecodeID').isNotNull())

In [0]:
yellow_store_and_fwd_flag_mode = yellow_df.select(F.mode('store_and_fwd_flag')).first()[0]
yellow_store_and_fwd_flag_mode

'N'

In [0]:
yellow_df = yellow_df.na.fill(value=yellow_store_and_fwd_flag_mode, subset=["store_and_fwd_flag"])

### Validating datasets size

In [0]:
check_removal_percentage(green_df, original_green_size)
check_removal_percentage(yellow_df, original_yellow_size)

--- Data Removal Report ---
Original size: 83,484,688
Current size : 77,860,217
Rows removed : 5,624,471
Percentage removed: 6.74%
[0% |███-----------------------------------------------| 100%]
----------------------------
--- Data Removal Report ---
Original size: 907,982,776
Current size : 840,946,411
Rows removed : 67,036,365
Percentage removed: 7.38%
[0% |███-----------------------------------------------| 100%]
----------------------------


## Combine datasets

Renaming pickup and dropoff dates

In [0]:
green_df = green_df.withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime')
green_df = green_df.withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

yellow_df = yellow_df.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime')
yellow_df = yellow_df.withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

Create trip type column in yellow taxis df with 0's

In [0]:
yellow_df = yellow_df.withColumn('trip_type',F.lit(0))

Create a flag column that highlights the taxi type (green or yellow)

In [0]:
green_df = green_df.withColumn('taxi_type',F.lit('green'))
yellow_df = yellow_df.withColumn('taxi_type',F.lit('yellow'))

Union of the two df's

In [0]:
taxis_df = yellow_df.unionByName(green_df)

### Combine with location data

In [0]:
location_df = spark.table(LOCATION_TABLE_PATH)

In [0]:
pickup_locations_alias = location_df.alias("pickup")
dropoff_locations_alias = location_df.alias("dropoff")

final_df = taxis_df.join(
    pickup_locations_alias,
    taxis_df.PULocationID == F.col("pickup.LocationID"),
    "left"
).join(
    dropoff_locations_alias,
    taxis_df.DOLocationID == F.col("dropoff.LocationID"),
    "left"
).select(
    *[c for c in taxis_df.columns if c not in ['PULocationID', 'DOLocationID']],
    F.col("pickup.Borough").alias("pickup_borough"),
    F.col("pickup.Zone").alias("pickup_zone"),
    F.col("dropoff.Borough").alias("dropoff_borough"),
    F.col("dropoff.Zone").alias("dropoff_zone")
)

### Adding extra filters

1. Not Unknown values in pickup_borough dropoff_borough
2. More or equal to 0 the total amount
3. Less or equal to 6 number of passagers


In [0]:
final_df = final_df.filter(F.col('pickup_borough')!='Unknown')
final_df = final_df.filter(F.col('dropoff_borough')!='Unknown')
final_df = final_df.filter(F.col('total_amount')>=0)
final_df = final_df.filter(F.col('passenger_count')<=6)

## Storing Taxis full table

In [0]:
final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TAXIS_TABLE_PATH)

In [0]:
final_taxis_df = spark.table(TAXIS_TABLE_PATH)
final_taxis_df.count()

904897502